In [3]:
!pip install ultralytics

In [5]:
from collections import defaultdict

import cv2
import numpy as np
from google.colab.patches import cv2_imshow
from ultralytics import YOLO
from google.colab import drive
drive.mount('/content/drive')

# Load the YOLOv8 model
model = YOLO('yolov8n.pt')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [55]:
import csv
import pandas as pd
import random
import datetime





# Open the video file
#video_path ='/content/drive/MyDrive/0_497Project/preliminarydata/results/clipped_street_crossing.mov'
video_path ='/content/drive/MyDrive/0_497Project/preliminarydata/ZOOM0007.mov'
#output_path = '/content/drive/MyDrive/0_497Project/preliminarydata/results/tracking_objects3.csv'

# Open CSV file to store box positions

counter_frames=0
cap = cv2.VideoCapture(video_path)

# Store the track history
#defaultdict that stores the history of tracked objects
track_history = defaultdict(lambda: [])
id_dict = {}
# Loop through the video frames
#It's a loop that processes each frame of the video until it reaches the end or the user presses
while cap.isOpened():
    # Read a frame from the video
    success, frame = cap.read()
    df = pd.DataFrame(columns=["start time", "end time"])

    if success:
        # Run YOLOv8 tracking on the frame, persisting tracks between frames
        #It retrieves the bounding boxes and track IDs of detected objects.
        results = model.track(frame, persist=True)
        if results[0].boxes.id != None:
          #print(results[0].boxes)
          # Get the boxes and track IDs
          boxes = results[0].boxes.xywh.cpu()
          track_ids = results[0].boxes.id.int().cpu().tolist()

          # Visualize the results on the frame
          annotated_frame = results[0].plot()
          #save tracking history
          if len(track_ids)==1:
            type_id = int(results[0].boxes[3].cls[0])
            if type_id==0:
              if id_dict.get(track_ids[0])==None:
                id_dict[track_ids[0]]=[counter_frames, counter_frames]
              else:
                id_dict[track_ids[0]][1]= counter_frames

          else:
            for i in range(len(track_ids)):
              type_id = int(results[0].boxes[i].cls[0])
              if type_id==0:
                if id_dict.get(track_ids[i])==None:
                  id_dict[track_ids[i]]=[counter_frames, counter_frames]
                else:
                  id_dict[track_ids[i]][1]= counter_frames

        else:
          continue

        # Plot the tracks
        for box, track_id in zip(boxes, track_ids):
            x, y, w, h = box
            track = track_history[track_id]
            track.append((float(x), float(y)))  # x, y center point
            if len(track) > 30:  # retain 90 tracks for 90 frames
                track.pop(0)

            # Draw the tracking lines
            points = np.hstack(track).astype(np.int32).reshape((-1, 1, 2))
            cv2.polylines(annotated_frame, [points], isClosed=False, color=(230, 230, 230), thickness=10)

        # Display the annotated frame
        #cv2_imshow( annotated_frame)#"YOLOv8 Tracking",

        # Break the loop if 'q' is pressed
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    else:
        # Break the loop if the end of the video is reached
        break
    counter_frames += 1
# Release the video capture object and close the display window
cap.release()
cv2.destroyAllWindows()





0: 384x640 3 persons, 1 car, 1 stop sign, 167.7ms
Speed: 7.5ms preprocess, 167.7ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 1 car, 1 stop sign, 179.0ms
Speed: 5.6ms preprocess, 179.0ms inference, 2.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 1 car, 1 stop sign, 175.6ms
Speed: 5.0ms preprocess, 175.6ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 1 car, 1 stop sign, 165.4ms
Speed: 4.5ms preprocess, 165.4ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 1 car, 1 stop sign, 173.8ms
Speed: 6.9ms preprocess, 173.8ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 1 car, 1 stop sign, 160.7ms
Speed: 4.6ms preprocess, 160.7ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 1 car, 1 stop sign, 183.8ms
Speed: 2.8ms preprocess, 183.8ms inference, 1.4ms postp

In [57]:
for i in id_dict.keys():
  df = df.append({"start time": id_dict[i][0], "end time": id_dict[i][1]}, ignore_index=True)
df.to_csv("/content/drive/MyDrive/0_497Project/preliminarydata/results/time_intervals.csv", index=False)


<ipython-input-57-81a8e84b9996>:2: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append({"start time": id_dict[i][0], "end time": id_dict[i][1]}, ignore_index=True)
<ipython-input-57-81a8e84b9996>:2: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append({"start time": id_dict[i][0], "end time": id_dict[i][1]}, ignore_index=True)
<ipython-input-57-81a8e84b9996>:2: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append({"start time": id_dict[i][0], "end time": id_dict[i][1]}, ignore_index=True)
<ipython-input-57-81a8e84b9996>:2: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append({"start time": id_dict[i][0],

In [62]:
counter_frames
df.iloc[0][1]

113

In [63]:

def extract_segments(video_path, df):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("Error: Could not open video.")
        return

    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))


    for i in range(df.shape[0]):
        start_frame = df.iloc[i][0]
        end_frame = df.iloc[i][1]

        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
        output_video = cv2.VideoWriter(f"segment_{i+1}.avi", fourcc, fps, (width, height))

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret or cap.get(cv2.CAP_PROP_POS_FRAMES) > end_frame:
                break
            output_video.write(frame)

        output_video.release()

    cap.release()

extract_segments(video_path, df)
